# 13. openFDA API EDA (T0-4)

**데이터 출처**: https://api.fda.gov (인증 불필요)  
**파이프라인 역할**: 부작용/라벨 구조 sanity check, 보고서 보조 참조

## EDA 체크리스트
- [ ] 주요 엔드포인트 응답 구조 확인 (`/drug/label`, `/drug/event`)
- [ ] 한국 성분명 50종 → FDA 라벨 검색 히트율
- [ ] 부작용 보고 건수 분포
- [ ] 산출물: `openfda_summary.json`

In [ ]:
import json, time
import requests
import pandas as pd
from pathlib import Path

ROOT = Path("../../").resolve()
INTERIM = ROOT / "data" / "interim"

BASE = "https://api.fda.gov"

In [ ]:
def openfda_get(endpoint, params, timeout=15):
    resp = requests.get(f"{BASE}{endpoint}", params=params, timeout=timeout)
    if resp.status_code == 404:
        return None
    resp.raise_for_status()
    return resp.json()

# 연결 확인
r = openfda_get("/drug/label.json", {"search": "openfda.generic_name:aspirin", "limit": 1})
print("총 aspirin 라벨:", r["meta"]["results"]["total"])

In [ ]:
# 라벨 주요 필드 구조 확인
result = r["results"][0]
print("최상위 키:", list(result.keys())[:20])

In [ ]:
# 한국 주요 성분명 50종 히트율 테스트
# 낱알식별 parquet이 있으면 실제 성분명 추출, 없으면 예시 목록 사용
nadal_path = INTERIM / "nadal_ident_clean.parquet"

if nadal_path.exists():
    df_nadal = pd.read_parquet(nadal_path)
    ingr_col = next((c for c in df_nadal.columns if "INGR" in c.upper()), None)
    if ingr_col:
        test_names = df_nadal[ingr_col].dropna().value_counts().head(50).index.tolist()
    else:
        test_names = ["아세트아미노펜", "이부프로펜", "아스피린", "메트포르민", "암로디핀"]
else:
    test_names = ["acetaminophen", "ibuprofen", "aspirin", "metformin", "amlodipine",
                  "atorvastatin", "omeprazole", "losartan", "metoprolol", "lisinopril"]

print(f"테스트 성분명 {len(test_names)}종")

In [ ]:
hit_results = {}
for name in test_names[:20]:  # rate limit 고려 20종만
    r = openfda_get("/drug/label.json", {
        "search": f"openfda.generic_name:{name}",
        "limit": 1
    })
    hit = bool(r and r.get("results"))
    hit_results[name] = hit
    time.sleep(0.2)

hit_rate = sum(hit_results.values()) / len(hit_results)
print(f"히트율: {hit_rate:.1%}")
for name, hit in hit_results.items():
    print(f"  {'✓' if hit else '✗'} {name}")

In [ ]:
# 부작용(adverse event) 건수 상위 성분
r_event = openfda_get("/drug/event.json", {
    "count": "patient.drug.openfda.generic_name.exact",
    "limit": 20
})
if r_event:
    top_ae = pd.DataFrame(r_event["results"])
    print(top_ae.head(10))

In [ ]:
summary = {
    "test_drug_count": len(hit_results),
    "hit_rate": float(hit_rate),
    "hit_details": hit_results
}
with open(INTERIM / "openfda_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("저장 완료")